# 단디봄 통합 MVP — 1단계

이 노트북은 백엔드, 복지사 앱, 사용자 앱을 한 Colab 런타임에 배치한 통합본입니다.

현재 적용 범위:
- 공통 라이브러리 설치를 한 번만 실행
- 백엔드 코드를 한 번만 정의
- 복지사·사용자 앱의 전역 변수, 라우터, 렌더 함수 이름 분리
- 두 앱은 각각 별도 Gradio URL로 실행

아직 적용하지 않은 범위:
- 복지사 화면 더미데이터의 DB 조회 연결
- 사용자 앱 더미 BackendAPI의 실제 백엔드 연결


In [107]:
%pip uninstall -y gradio gradio_client
%pip install -q "gradio==6.5.1" langchain langchain-openai langchain-chroma langchain-community langchain-huggingface chromadb sentence-transformers pandas tqdm ragas datasets


Found existing installation: gradio 6.5.1
Uninstalling gradio-6.5.1:
  Successfully uninstalled gradio-6.5.1
Found existing installation: gradio_client 2.0.3
Uninstalling gradio_client-2.0.3:
  Successfully uninstalled gradio_client-2.0.3


## 실행 준비 (ver3 · DB 연동)

설치 셀 실행 후 런타임을 한 번 다시 시작하고 아래 셀을 순서대로 실행합니다.

`/content`에 업로드할 파일:
- `dandibom.db` 또는 `dandibom(1).db`
- `dialects.zip`
- `/content/assets/dandibom_logo.png`

준비 셀이 DB 이름 통일과 사투리 ZIP 압축 해제를 자동으로 처리합니다.


# 1. 공통 백엔드

백엔드 함수와 VectorDB·LLM 코드는 이 구역에서 한 번만 정의됩니다.


In [108]:
from google.colab import userdata
import os

from pathlib import Path
import json
import re
import pandas as pd
from tqdm.auto import tqdm
import sqlite3
import datetime

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

from openai import OpenAI

In [109]:
# Colab 파일 경로 준비
import shutil
import zipfile

DB_CANDIDATES = [Path("/content/dandibom.db"), Path("/content/dandibom(1).db")]
DB_PATH = next((path for path in DB_CANDIDATES if path.exists()), None)

if DB_PATH is None:
    raise FileNotFoundError(
        "DB 파일이 없습니다. dandibom.db 또는 dandibom(1).db를 /content에 업로드해주세요."
    )

# 모든 백엔드 함수가 동일한 고정 경로를 사용하도록 이름을 통일합니다.
canonical_db_path = Path("/content/dandibom.db")
if DB_PATH != canonical_db_path:
    shutil.copy2(DB_PATH, canonical_db_path)
DB_PATH = canonical_db_path

DIALECT_DIR = Path("/content/dialects")
DIALECT_ZIP = Path("/content/dialects.zip")
if DIALECT_ZIP.exists():
    DIALECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DIALECT_ZIP) as archive:
        archive.extractall("/content")

if not DIALECT_DIR.exists() or not list(DIALECT_DIR.glob("*.json")):
    raise FileNotFoundError(
        "사투리 JSON이 없습니다. dialects.zip을 /content에 업로드하거나 "
        "/content/dialects 폴더에 JSON 파일을 넣어주세요."
    )

if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

def get_db_connection():
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

print(f"DB 연결 준비 완료: {DB_PATH}")
print(f"사투리 JSON 준비 완료: {len(list(DIALECT_DIR.glob('*.json')))}개")


DB 연결 준비 완료: /content/dandibom.db
사투리 JSON 준비 완료: 2개


# DB 사용 함수
### 사용가능 API(함수) 정리

- def chat_with_llm(user_id, user_query, k=5, verbose=True)
- def load_user_data_from_db(db_path='dandibom.db'):
- def save_conversation_to_db(user_id, role, message, db_path='dandibom.db'):
- def save_medication(user_id, medicine):
- def get_medication(user_id):
- def save_condition(user_id, meal, sleep, mood, pain):
- def get_latest_condition(user_id):
- def chat_analysis(user_id, chat):
- def save_alert(user_id, cause, level, status):
- def get_alert(user_id):

In [110]:
def load_user_data_from_db(db_path=None):
    """SQLite DB에서 사용자 및 조건 데이터를 로드합니다."""
    conn = sqlite3.connect(str(db_path or DB_PATH))

    # Updated schema for empty dataframes
    users_cols = ['user_id', 'role', 'name', 'code']
    conditions_cols = ['cond_id', 'user_id', 'meal', 'sleep', 'mood', 'pain', 'medicine', 'recorded_at']

    try:
        users_df = pd.read_sql_query("SELECT * FROM users", conn)
        # Ensure user_id is int type after loading
        if 'user_id' in users_df.columns:
            users_df['user_id'] = pd.to_numeric(users_df['user_id'], errors='coerce').astype('Int64')
    except pd.io.sql.DatabaseError:
        print("Warning: 'users' table not found or empty. Creating an empty DataFrame.")
        users_df = pd.DataFrame(columns=users_cols)

    try:
        conditions_df = pd.read_sql_query("SELECT * FROM conditions", conn)
        # Ensure user_id is int type after loading
        if 'user_id' in conditions_df.columns:
            conditions_df['user_id'] = pd.to_numeric(conditions_df['user_id'], errors='coerce').astype('Int64')
    except pd.io.sql.DatabaseError:
        print("Warning: 'conditions' table not found or empty. Creating an empty DataFrame.")
        conditions_df = pd.DataFrame(columns=conditions_cols)

    conn.close()
    return users_df, conditions_df

users_df, conditions_df = load_user_data_from_db()

print("Loaded users data:")
display(users_df.head())
print("Loaded conditions data:")
display(conditions_df.head())

Loaded users data:


,user_id,role,name,code
0,1,대상자,김순자,A3F7
1,2,보호자,박지훈,A4E8
2,3,복지사,이경미,A5G9
3,4,대상자,박막례,B2K9
4,5,대상자,이팔복,C7M1


Loaded conditions data:


,cond_id,user_id,meal,sleep,mood,pain,recorded_at
0,1,1,"{""breakfast"": ""y"", ""lunch"": ""y"", ""dinner"": ""y""}",보통,보통,없음,2026-07-11 09:10:00
1,2,1,"{""breakfast"": ""y"", ""lunch"": ""y"", ""dinner"": ""y""}",보통,보통,약간,2026-07-12 09:05:00
2,3,1,"{""breakfast"": ""y"", ""lunch"": ""n"", ""dinner"": ""n""}",잠 설침,불쾌함,약간,2026-07-13 09:20:00
3,4,1,"{""breakfast"": ""n"", ""lunch"": ""y"", ""dinner"": ""n""}",잠 설침,불쾌함,약간,2026-07-14 09:15:00
4,5,4,"{""breakfast"": ""y"", ""lunch"": ""y"", ""dinner"": ""y""}",잘잠,상쾌함,없음,2026-07-15 08:40:00


In [111]:
def save_conversation_to_db(user_id, role, message, db_path=None):
    """대화 내역을 conversations 테이블에 저장합니다."""
    conn = sqlite3.connect(str(db_path or DB_PATH), timeout=10)
    cursor = conn.cursor()
    created_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            "INSERT INTO conversations (user_id, role, message, created_at) VALUES (?, ?, ?, ?)",
            (user_id, role, message, created_at)
        )
        conn.commit()
        if role == 'user':
            print(f"User message for user_id {user_id} saved to DB.")
        else:
            print(f"Assistant message for user_id {user_id} saved to DB.")
    except Exception as e:
        print(f"Error saving conversation to DB: {e}")
    finally:
        conn.close()

## 복용 데이터

```
CREATE TABLE IF NOT EXISTS medications (
    cond_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    medicine VARCHAR(255) NOT NULL,
    recorded_at DATETIME NOT NULL,  -- 기록 날짜
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
```

### 복용 데이터 추가

In [112]:
def save_medication(user_id, medicine):
    conn = get_db_connection()
    cursor = conn.cursor()

    recorded_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            "INSERT INTO medications (user_id, medicine, recorded_at) VALUES (?, ?, ?)",
            (user_id, medicine, recorded_at)
        )
        conn.commit()
        print(f"약복용 데이터 저장 완료 '{medicine}'. 사용자: {user_id}.")
    except sqlite3.Error as e:
        print(f"약 복용 데이터 저장 오류: {e}")
        conn.rollback()
    finally:
        conn.close()

def save_morning_medication(user_id, status):
    if status == "✓먹었어요":
        save_medication(user_id, "아침약")
        return "아침약 복용을 기록했습니다."

    return "아직 복용하지 않은 것으로 확인했습니다."

### 복용 데이터 조회

In [113]:
def get_medication(user_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM medications WHERE user_id = ? ORDER BY recorded_at DESC",
            (user_id,)
        )
        medications = cursor.fetchall()
        if medications:
            print(f"사용자 {user_id}의 복용 데이터 불러오기 완료.")
            return medications
        else:
            print(f"사용자 {user_id}에 대한 복용 데이터 없음.")
            return None
    except sqlite3.Error as e:
        print(f"복용 데이터 불러오기 오류: {e}")
        return None
    finally:
        conn.close()

## 컨디션 데이터

```
CREATE TABLE IF NOT EXISTS conditions (
    cond_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    meal VARCHAR(255) NOT NULL,
    sleep VARCHAR(255) NOT NULL,
    mood VARCHAR(255) NOT NULL,
    pain VARCHAR(255) NOT NULL,
    recorded_at DATETIME NOT NULL,  -- 기록 날짜
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
```

### 컨디션 데이터 추가

In [114]:
def save_condition(user_id, meal, sleep, mood, pain):
    conn = get_db_connection()
    cursor = conn.cursor()

    recorded_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            """INSERT INTO conditions
            (user_id, meal, sleep, mood, pain, recorded_at)
            VALUES (?, ?, ?, ?, ?, ?)""",
            (user_id, meal, sleep, mood, pain, recorded_at)
        )
        conn.commit()
        print(f"컨디션 데이터 저장 완료. 사용자 아이디: {user_id}.")
    except sqlite3.Error as e:
        print(f"컨디션 데이터 저장 오류: {e}")
        conn.rollback()
    finally:
        conn.close()

### 사용자별 최근 컨디션 데이터 조회

In [115]:
def get_latest_condition(user_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM conditions WHERE user_id = ? ORDER BY recorded_at DESC LIMIT 1",
            (user_id,)
        )
        condition = cursor.fetchone()
        if condition:
            print(f"사용자 {user_id} 최신 컨디션 데이터 불러오기 완료.")
            return dict(condition) # Convert row object to dictionary
        else:
            print(f"사용자 {user_id}에 대한 컨디션 데이터 없음.")
            return None
    except sqlite3.Error as e:
        print(f"최신 컨디션 데이터 불러오기 오류: {e}")
        return None
    finally:
        conn.close()

## alerts 데이터

```
CREATE TABLE IF NOT EXISTS alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    cause TEXT NOT NULL,  -- 이상신호 근거
    level VARCHAR(10) NOT NULL CHECK(level IN ('낮음', '중간', '높음')),
    status VARCHAR(10) NOT NULL CHECK(status IN ('대기', '확인')),
    created_at date NOT NULL,  -- 이상신호 발생 시각
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
```

### alert 함수에 데이터 추가

#### 대화 내용 분석 함수

In [116]:
# OpenAI API 키 불러오기
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

def chat_analysis(user_id, chat):
    system_prompt = """
    너는 사용자의 대화 내용을 분석하는 챗봇이야.
    사용자가 말하는 내용에서 무엇 때문에 불편해 하는지, 그리고 그 정도가 어느 수준인지 파악해야 해.
    분석 결과는 JSON 형식으로만 응답해야 하며, 다음 형식을 따라야 해:
    {
        "cause": "불편함의 원인을 한 문장으로 요약",
        "level": "불편함의 수준 (낮음, 중간, 높음 중 하나)"
    }
    불편함이 없는 일반적인 대화인 경우, cause는 '특이사항 없음'으로, level은 '낮음'으로 설정해.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={ "type": "json_object" },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": chat}
            ]
        )
        result = response.choices[0].message.content

        parsed_result = json.loads(result)
        cause = parsed_result.get('cause', '분석 불가')
        level = parsed_result.get('level', '낮음')
        print(f"사용자 {user_id}의 대화 분석 완료: 원인 - {cause}, 수준 - {level}")
        return cause, level
    except Exception as e:
        print(f"대화 분석 중 오류 발생: {e}")
        return "분석 오류", "낮음"

#### 데이터 저장

In [117]:
def save_alert(user_id, cause, level, status):
    conn = get_db_connection()
    cursor = conn.cursor()
    created_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            "INSERT INTO alerts (user_id, cause, level, status, created_at) VALUES (?, ?, ?, ?, ?)",
            (user_id, cause, level, status, created_at)
        )
        conn.commit()
        print(f"사용자 {user_id}의 위험 신호 데이터 추가. Cause: {cause}, Level: {level}, Status: {status}")
    except sqlite3.Error as e:
        print(f"위험 신호 데이터 저장 오류: {e}")
        conn.rollback()
    finally:
        conn.close()

### 사용자별 alerts 데이터 조회

In [118]:
def get_alert(user_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM alerts WHERE user_id = ? ORDER BY created_at DESC",
            (user_id,)
        )
        alerts = cursor.fetchall()
        if alerts:
            print(f"사용자 {user_id}의 위험 신호 데이터 불러오기 완료.")
            return alerts
        else:
            print(f"사용자 {user_id}에 대한 위험 신호 데이터 없음.")
            return None
    except sqlite3.Error as e:
        print(f"위험 신호 데이터 불러오기 오류: {e}")
        return None
    finally:
        conn.close()

# 텍스트 임베딩

후보
- text-embedding-3-small
- nlpai-lab/KURE-v1
- BAAI/bge-m3

In [119]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 경상도 사투리 VectorDB 구축

목표:
1. 특정 폴더 안의 경상도 사투리 JSON 파일들을 일괄 조회한다.
2. `transcription.sentences` 안의 `sentenceId`, `dialect`, `standard`, `pronunciation`을 추출한다.
3. `standard` 결측, `dialect` 결측, `dialect == standard` 데이터는 제거한다. (즉, 값이 없거나 사투리가 아닌 것은 취급하지 않는다)
4. 전체 파일 기준으로 중복되는 `dialect-standard` 쌍을 제거한다.
5. 정제 결과를 CSV로 저장한다.
6. LangChain `Document`로 변환한 뒤 Chroma VectorDB를 구축한다.

이번 MVP에서는 별도 청킹을 하지 않는다. JSON의 `sentences` row 하나가 이미 하나의 문장 단위 Document이다.


## 1. 경로 설정

1. 우선 경상도 사투리 파일을 저장할 'dialects'라는 경로를 생성한다.

2. 생성된 경로에 전달받은 사투리 JSON 파일을 넣는다.
- talk_set3_collectorgs161_speakergs1589_speakergs1590_6_0_116.json
- talk_set3_collectorgs164_speakergs1380_speakergs1381_3_0_98.json

In [120]:
# dialects 경로 생성
if not os.path.exists('/content/dialects'):
    os.makedirs('/content/dialects')
    print("'/content/dialects' directory created.")
else:
    print("'/content/dialects' directory already exists.")

'/content/dialects' directory already exists.


In [121]:
# TODO: 본인 JSON 파일 폴더 경로로 수정
JSON_DIR = DIALECT_DIR  # JSON_DIR = Path("/content/drive/MyDrive/gyeongsang_json_923")

# 정제 CSV 저장 경로
OUTPUT_CSV_PATH = Path("./gyeongsang_dialect_cleaned.csv")

# Chroma VectorDB 저장 경로
PERSIST_DIR = Path("./chroma_gyeongsang_dialect_db")

# Chroma collection 이름
COLLECTION_NAME = "gyeongsang_dialect"

## 2. 전처리

현재 프로젝트에서는 `transcription.sentences`만 사용한다.

제외 조건:
- `dialect`가 null 또는 빈 문자열
- `standard`가 null 또는 빈 문자열
- `dialect == standard`
- 너무 짧은 문장
- 너무 긴 문장
- 전체 파일 기준 중복 `dialect-standard` 쌍


In [122]:
def normalize_text(value):
    """None, 숫자, 문자열 입력을 안전하게 문자열로 변환하고 공백을 정리한다."""
    if value is None:
        return ""
    return str(value).strip()


def is_valid_pair(dialect, standard, min_len=2, max_len=200):
    """VectorDB에 저장할 수 있는 dialect-standard 쌍인지 판단한다."""
    dialect = normalize_text(dialect)
    standard = normalize_text(standard)

    if not dialect or not standard:
        return False

    if dialect == standard:
        return False

    if len(dialect) < min_len or len(standard) < min_len:
        return False

    if len(dialect) > max_len or len(standard) > max_len:
        return False

    return True


def extract_rows_from_json(json_path):
    """JSON 파일 1개에서 유효한 sentence row들을 추출한다."""
    json_path = Path(json_path)

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"[로드 실패] {json_path.name}: {e}")
        return []

    file_name = normalize_text(data.get("fileName")) or json_path.stem

    transcription = data.get("transcription", {})
    if not isinstance(transcription, dict):
        return []

    sentences = transcription.get("sentences", [])
    if not isinstance(sentences, list):
        return []

    rows = []

    for sent in sentences:
        if not isinstance(sent, dict):
            continue

        dialect = normalize_text(sent.get("dialect"))
        standard = normalize_text(sent.get("standard"))
        pronunciation = normalize_text(sent.get("pronunciation"))
        sentence_id = normalize_text(sent.get("sentenceId"))
        speaker_id = normalize_text(sent.get("speakerId"))

        if not is_valid_pair(dialect, standard):
            continue

        rows.append({
            "source_file": file_name,
            "sentence_id": sentence_id,
            "speaker_id": speaker_id,
            "dialect": dialect,
            "standard": standard,
            "pronunciation": pronunciation,
        })

    return rows

## 3. 폴더 내 JSON 전체 파싱 및 정제

923개 JSON 파일 전체를 읽고, 유효한 `dialect-standard` 쌍만 남긴다.


In [123]:
def build_clean_dataframe(json_dir):
    """폴더 안의 모든 JSON 파일을 읽어 정제 DataFrame을 만든다."""
    json_dir = Path(json_dir)

    if not json_dir.exists():
        raise FileNotFoundError(f"JSON_DIR이 존재하지 않습니다: {json_dir}")

    json_files = sorted(json_dir.glob("*.json"))

    if not json_files:
        raise FileNotFoundError(f"폴더 안에 JSON 파일이 없습니다: {json_dir}")

    all_rows = []

    for json_path in tqdm(json_files, desc="JSON 파일 처리 중"):
        rows = extract_rows_from_json(json_path)
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)

    if df.empty:
        print("유효한 dialect-standard row가 없습니다.")
        return df

    before_dedup = len(df)

    # 전체 파일 기준 dialect-standard 중복 제거
    df = df.drop_duplicates(subset=["dialect", "standard"]).reset_index(drop=True)

    after_dedup = len(df)

    print(f"JSON 파일 수: {len(json_files):,}")
    print(f"중복 제거 전 row 수: {before_dedup:,}")
    print(f"중복 제거 후 row 수: {after_dedup:,}")

    return df


clean_df = build_clean_dataframe(JSON_DIR)
clean_df.head()


JSON 파일 처리 중:   0%|          | 0/2 [00:00<?, ?it/s]

JSON 파일 수: 2
중복 제거 전 row 수: 6
중복 제거 후 row 수: 6


,source_file,sentence_id,speaker_id,dialect,standard,pronunciation
0,talk_set3_collectorgs161_speakergs1589_speaker...,1.0,speakergs1589,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요,먹을 때 시원한 그런 음식을 먹으면은 어 조은데예
1,talk_set3_collectorgs161_speakergs1589_speaker...,3.0,speakergs1589,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...,그 인제 여름에 이렇게 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다...,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...
2,talk_set3_collectorgs161_speakergs1589_speaker...,6.0,speakergs1589,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 하는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다
3,talk_set3_collectorgs164_speakergs1380_speaker...,4.0,speakergs1380,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...
4,talk_set3_collectorgs164_speakergs1380_speaker...,5.0,speakergs1380,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...,산에 막 소 먹이러 다니고 그래서 시골에 대한 그런 거 또 아침에 일어나서 또 시골...,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...


## 4. 정제 결과 확인 및 CSV 저장


In [124]:
print(clean_df.shape)
display(clean_df.head(20))

# 정제 결과 확인을 위한 csv 파일 생성
if not clean_df.empty:
    clean_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"정제 CSV 저장 완료: {OUTPUT_CSV_PATH.resolve()}")

(6, 6)


,source_file,sentence_id,speaker_id,dialect,standard,pronunciation
0,talk_set3_collectorgs161_speakergs1589_speaker...,1.0,speakergs1589,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요,먹을 때 시원한 그런 음식을 먹으면은 어 조은데예
1,talk_set3_collectorgs161_speakergs1589_speaker...,3.0,speakergs1589,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...,그 인제 여름에 이렇게 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다...,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...
2,talk_set3_collectorgs161_speakergs1589_speaker...,6.0,speakergs1589,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 하는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다
3,talk_set3_collectorgs164_speakergs1380_speaker...,4.0,speakergs1380,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...
4,talk_set3_collectorgs164_speakergs1380_speaker...,5.0,speakergs1380,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...,산에 막 소 먹이러 다니고 그래서 시골에 대한 그런 거 또 아침에 일어나서 또 시골...,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...
5,talk_set3_collectorgs164_speakergs1380_speaker...,6.0,speakergs1380,초가 진짜 초가집이어 갖고 막 지붕 새로 이고 막 이런 그런 어릴 적에 그 추억이 ...,초가 진짜 초가집이어 갖고 막 지붕 새로 얹고 막 이런 그런 어릴 적에 그 추억이 ...,초가 진짜 초가집이어 갖고 막 지붕 새로 이고 막 이런 그런 어릴 적에 그 추억이 ...


정제 CSV 저장 완료: /content/gyeongsang_dialect_cleaned.csv


## 5. Document 생성

중요:
- `page_content`에는 `dialect`만 넣는다.
- `standard`, `sentence_id`, `source_file`은 metadata에 넣는다.
- 이렇게 해야 사용자 입력 사투리와 `dialect`가 직접 비교되어 검색된다.


In [125]:
def dataframe_to_documents(df):
    """정제 DataFrame을 LangChain Document 리스트로 변환한다."""
    documents = []

    for idx, row in df.iterrows():
        dialect = row["dialect"]
        standard = row["standard"]

        metadata = {
            "standard": standard,
            "sentence_id": normalize_text(row.get("sentence_id")),
            "source_file": normalize_text(row.get("source_file")),
            "speaker_id": normalize_text(row.get("speaker_id")),
            "pronunciation": normalize_text(row.get("pronunciation")),
            "row_index": int(idx),
        }

        documents.append(
            Document(
                page_content=dialect,
                metadata=metadata
            )
        )

    return documents


documents = dataframe_to_documents(clean_df)
print(f"Document 수: {len(documents):,}")

if documents:
    print("[Document 예시]")
    print("page_content:", documents[0].page_content)
    print("metadata:", documents[0].metadata)

Document 수: 6
[Document 예시]
page_content: 먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예
metadata: {'standard': '먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요', 'sentence_id': '1.0', 'source_file': 'talk_set3_collectorgs161_speakergs1589_speakergs1590_6_0_116', 'speaker_id': 'speakergs1589', 'pronunciation': '먹을 때 시원한 그런 음식을 먹으면은 어 조은데예', 'row_index': 0}


## 6. VectorDB 구축 (ChromaDB)

In [126]:
if not documents:
    raise ValueError("VectorDB에 저장할 Document가 없습니다.")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIR),
    collection_name=COLLECTION_NAME,
)

print(f"Chroma VectorDB 구축 완료: {PERSIST_DIR.resolve()}")
print(f"Collection name: {COLLECTION_NAME}")

Chroma VectorDB 구축 완료: /content/chroma_gyeongsang_dialect_db
Collection name: gyeongsang_dialect


## 7. 검색 테스트

사용자 입력과 유사한 사투리 문장을 검색한다.


In [127]:
def search_dialect_examples(query, k=5):
    """사용자 입력과 유사한 사투리 예시를 검색한다."""
    results = vectorstore.similarity_search(query, k=k)

    formatted = []

    for i, doc in enumerate(results, start=1):
        formatted.append({
            "rank": i,
            "dialect": doc.page_content,
            "standard": doc.metadata.get("standard", ""),
            "sentence_id": doc.metadata.get("sentence_id", ""),
            "source_file": doc.metadata.get("source_file", ""),
        })

    return pd.DataFrame(formatted)


test_query = "밥도 안 묵고 속이 답답하데이"
search_result_df = search_dialect_examples(test_query, k=5)
display(search_result_df)

,rank,dialect,standard,sentence_id,source_file
0,1,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,4.0,talk_set3_collectorgs164_speakergs1380_speaker...
1,2,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,4.0,talk_set3_collectorgs164_speakergs1380_speaker...
2,3,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,4.0,talk_set3_collectorgs164_speakergs1380_speaker...
3,4,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,4.0,talk_set3_collectorgs164_speakergs1380_speaker...
4,5,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,4.0,talk_set3_collectorgs164_speakergs1380_speaker...


## 8. LLM 프롬프트에 넣을 사투리 참고자료 포맷 만들기

이번 MVP에서는 LLM을 한 번만 호출할 예정이므로, 검색된 `dialect-standard` 쌍을 프롬프트에 삽입한다.


In [128]:
def format_dialect_context(query, k=5):
    """LLM 프롬프트에 넣을 사투리-표준어 참고자료 문자열 생성"""
    docs = vectorstore.similarity_search(query, k=k)

    lines = []

    for i, doc in enumerate(docs, start=1):
        dialect = doc.page_content
        standard = doc.metadata.get("standard", "")

        lines.append(f"\n\n{i}. \n방언: {dialect} \n표준어: {standard}")

    return "".join(lines)


dialect_context = format_dialect_context(test_query, k=5)

print("[사투리-표준어 참고자료]")
print(dialect_context)

[사투리-표준어 참고자료]


1. 
방언: 또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 하면 막 소미기로 다녔어요 
표준어: 또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 하면 막 소 먹이러 다녔어요

2. 
방언: 또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 하면 막 소미기로 다녔어요 
표준어: 또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 하면 막 소 먹이러 다녔어요

3. 
방언: 또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 하면 막 소미기로 다녔어요 
표준어: 또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 하면 막 소 먹이러 다녔어요

4. 
방언: 또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 하면 막 소미기로 다녔어요 
표준어: 또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 하면 막 소 먹이러 다녔어요

5. 
방언: 또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 하면 막 소미기로 다녔어요 
표준어: 또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 하면 막 소 먹이러 다녔어요


## 9. 프롬프트

**따뜻하고 사려 깊은 태도** — 어르신의 안부를 살피고 필요한 경우 도움을 제안합니다.

프롬프트 수정 필요 : 사용자 개인화 필요

- 특성 사용자를 구별하고, 해당 사용자의 정보에 기반해서 답변을 출력하도록 해야 함

### 시스템 프롬프트

In [129]:
SYSTEM_PROMPT = """
너는 어르신을 오랫동안 곁에서 모셔온, 정중하고 다정한 AI 집사이자 독거노인 안부 도우미 챗봇이야.

사용자는 경상도 사투리를 사용할 수 있어.
제공되는 [사투리-표준어 참고자료]를 참고해서 사용자 입력의 의미를 이해한 뒤 답변해줘.

# 지켜야 할 것 (DO)
- 항상 **정중한 존댓말**. "~하셨어요?", "~드릴게요", "~이십니다"
- **이미 알고 있다는 태도** 이전 대화와 [이전 건강 이슈 메모리]를 참고하여 맥락에 맞게 답변
- **재촉하지 않기** — 답 안 하셔도 부담 주지 않고 부드럽게
- **과하지 않은 따뜻함** — 품위 있게. 걱정은 표현하되 호들갑 X
- 어르신 사투리는 **알아듣되**, AI 응답은 **표준어 존댓말**
- 답변은 짧고 이해하기 쉽게

# 하지 말 것 (DON'T)
- 반말·동년배 말투 ("~했어?", "~하자") — 절대 금지
- 의료 단정 ("병원 가세요", "OO병입니다") — 비진단 원칙
- 약 복용, 치료법, 의학적 판단 직접 지시 ("타이레놀 섭취하세요")
- 기계적·사무적 응대 ("입력을 확인했습니다")
- 오글거리는 과잉 애정 표현

# 말투 예시 대비
| 상황 | 안 좋은 예 | 좋은 예 |
|---|---|---|
| 인사 | "안녕! 뭐해?" | "어르신, 오늘 하루 잘 보내고 계신가요?" |
| 걱정 | "헐 괜찮아?? 병원 가!" | "저런, 걱정입니다. 언제부터 그러셨어요?" |
| 확인 | "식사 입력 완료" | "식사 잘 챙기셨다니 다행입니다." |

# 건강 이슈 기억 규칙
- 사용자의 발화에서 건강, 식사, 복약, 수면, 정서, 낙상, 통증, 호흡 곤란 등 안부 관리상 중요한 이슈가 있는지 판단한다.
- [이전 건강 이슈 메모리]에 현재 사용자 입력과 관련된 내용이 있다면 자연스럽게 참고해서 답변한다.
- 단, 이전 건강 이슈를 무리하게 끌어오지 않는다.
- 사용자가 "아직도", "계속", "오늘도", "또", "그때처럼", "여전히" 같은 표현을 쓰면 최근 active 건강 이슈와 연결해서 이해할 수 있는지 검토한다.
- 통증, 어지러움, 식사 불량, 수면 문제, 낙상, 우울감, 호흡 곤란은 중요한 안부 이슈로 보고 조심스럽게 확인 질문을 한다.
- 의료 진단이나 병명은 단정하지 않는다.
- 챗봇의 역할은 진단이 아니라 안부 확인, 정서적 지지, 위험 신호 감지, 필요 시 보호자/복지사/119 연결 권유이다.
- 가벼운 불편 표현에는 쉬기, 무리하지 않기, 오래 서 있지 않기 등 일반적인 생활 주의 수준으로 답한다.
- 불편함이 계속되거나 심해진다고 말하면 보호자나 복지사에게 알려도 되는지 부드럽게 묻는다.
- 응급 위험 신호가 명시되면 병명을 추정하지 말고 즉시 119 연락 또는 주변 사람의 도움을 권유한다.
  예: 호흡 곤란, 심한 가슴 통증, 의식 저하, 갑작스러운 마비, 말 어눌함, 낙상 후 움직이기 어려움, 극심한 통증, 자해·죽고 싶다는 표현.

# 출력 규칙
- 최종 출력은 반드시 JSON 객체 하나만 반환한다.
- JSON 앞뒤에 설명문, 마크다운 코드블록, 주석을 붙이지 않는다.
- 사용자가 보는 실제 답변은 answer 필드에만 작성한다.
""".strip()


### 사용자 프롬프트

In [130]:
def build_user_prompt(
    user_input,
    dialect_context,
    user_name='어르신',
    user_conditions='특별한 건강 이슈 없음',
    health_memory_context="이전 건강 이슈 메모리 없음",
    recent_chat_history="최근 대화 없음"
):
    return f"""
너와 대화 중인 사용자는 '{user_name}' 어르신입니다.
{user_name} 어르신은 현재 다음과 같은 건강 이슈가 있습니다: {user_conditions}

[사투리-표준어 참고자료]
{dialect_context}

[이전 건강 이슈 메모리]
{health_memory_context}

[최근 대화]
{recent_chat_history}

[현재 사용자 입력]
{user_input}

위 정보를 바탕으로 사용자 입력을 이해하고, 독거노인 안부 도우미 챗봇으로서 답변해줘.

반드시 아래 JSON 형식으로만 출력해.
{{
  "answer": "사용자에게 보여줄 최종 답변",
  "health_memory_update": {{
    "should_save": true,
    "category": "health | meal | sleep | medication | emotion | safety | daily_life | social | appointment | other | none",
    "summary": "나중에 참고할 수 있는 한 문장 요약. 저장할 내용이 없으면 빈 문자열",
    "detail": {{
      "keywords": ["기억할 핵심 키워드"],
      "time_expr": "언제부터/언제 있었는지. 없으면 빈 문자열",
      "emotion": "감정 상태. 없으면 빈 문자열",
      "need_follow_up": true,
      "risk_level": "none | low | medium | high"
    }},
    "status": "active | resolved | unknown"
  }}
}}

저장할 건강 이슈가 없으면 health_memory_update.should_save는 false로 하고, category는 "none"으로 출력해.
""".strip()


# Retriever

In [131]:
# VectorDB 리트리버 사용

retriever = vectorstore.as_retriever()

# LLM

In [132]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)

# 최종 RAG 챗봇 실행 함수: Health Memory 포함

아래 셀부터는 기존 `qa_chain`, `qa_chain_with_memory`, `CareMemory`/`HealtheMemory` 혼합 코드를 쓰지 않고, `chat_with_llm()` 하나로 통일한다.

구조:
1. 사투리 VectorDB 검색
2. 이전 건강 이슈 메모리 검색
3. 최근 대화 기록 삽입
4. LLM 1회 호출
5. JSON 파싱
6. 답변 출력
7. 건강 이슈 저장


In [133]:
# 최근 대화 기록을 저장할 메모리 설정
# 최근 대화는 자연스러운 대화 흐름을 위한 보조 메모리이다.
memory = InMemoryChatMessageHistory(
    k=5,
    return_messages=True,
    memory_key="chat_history",
    input_key="user_input"
)

print("최근 대화 메모리 설정 완료")

최근 대화 메모리 설정 완료


In [134]:
class HealthMemory:
    """건강 이슈 중심 장기 메모리.

    최근 5개 대화와 별도로, 건강/식사/수면/복약/정서/안전 등
    안부 관리상 다시 참고할 만한 내용을 저장한다.
    """

    def __init__(self):
        self.memories = []

    def add_memory(self, user_input, assistant_answer, memory_update):
        self.memories.append({
            "created_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
            "category": memory_update.get("category", "none"),
            "summary": memory_update.get("summary", ""),
            "detail": memory_update.get("detail", {}),
            "status": memory_update.get("status", "unknown"),
            "original_input": user_input,
            "assistant_answer": assistant_answer,
        })

    def retrieve_related(self, user_input, max_results=5):
        """현재 입력과 관련 있는 건강 이슈 메모리를 단순 점수 기반으로 검색한다."""
        related = []

        for memory_item in self.memories:
            score = 0

            summary = memory_item.get("summary", "")
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            category = memory_item.get("category", "")
            status = memory_item.get("status", "")

            # 1. LLM이 뽑은 핵심 키워드가 현재 입력에 다시 등장하면 강하게 연결
            for keyword in keywords:
                keyword = str(keyword).strip()
                if keyword and keyword in user_input:
                    score += 3

            # 2. summary 안의 의미 있는 토큰이 현재 입력에 포함되면 약하게 연결
            for token in summary.split():
                token = token.strip(".,!?~…은는이가을를에의도만고")
                if len(token) >= 2 and token in user_input:
                    score += 1

            # 3. 이어지는 표현이면 active 이슈와 연결 가능성을 높임
            if any(word in user_input for word in ["아직도", "계속", "오늘도", "또", "그때처럼", "여전히"]):
                if status == "active":
                    score += 2

            # 4. category 자체가 입력에 직접 들어오는 경우는 드물지만 보조 점수로 둠
            if category and category != "none" and category in user_input:
                score += 1

            if score > 0:
                related.append((score, memory_item))

        related = sorted(related, key=lambda x: x[0], reverse=True)
        return [item for score, item in related[:max_results]]

    def format_for_prompt(self, related_memories):
        if not related_memories:
            return "이전 건강 이슈 메모리 없음"

        lines = []
        for i, memory_item in enumerate(related_memories, start=1):
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            lines.append(
                f"{i}. 날짜: {memory_item['created_at']} / "
                f"분류: {memory_item['category']} / "
                f"요약: {memory_item['summary']} / "
                f"키워드: {', '.join(map(str, keywords))} / "
                f"상태: {memory_item['status']}"
            )
        return "\n".join(lines)

    def to_dataframe(self):
        return pd.DataFrame(self.memories)


health_memory = HealthMemory()
print("건강 이슈 메모리 설정 완료")


건강 이슈 메모리 설정 완료


In [135]:
def format_recent_chat_history(memory):

    """ConversationBufferWindowMemory의 최근 대화 기록을 문자열로 변환한다."""
    memory_vars = memory.messages
    messages = memory.messages

    if not messages:
        return "최근 대화 없음"

    lines = []
    for msg in messages:
        role = "사용자" if msg.type == "human" else "AI"
        lines.append(f"{role}: {msg.content}")

    return "\n".join(lines)


def parse_llm_json(raw_output):
    """LLM이 반환한 JSON 문자열을 안전하게 파싱한다."""
    text = raw_output.strip()

    # 혹시 ```json ... ``` 형태로 출력될 경우 제거
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return {
            "answer": raw_output,
            "health_memory_update": {
                "should_save": False,
                "category": "none",
                "summary": "",
                "detail": {
                    "keywords": [],
                    "time_expr": "",
                    "emotion": "",
                    "need_follow_up": False,
                    "risk_level": "none"
                },
                "status": "unknown"
            }
        }

    # 필수 키가 없을 경우 기본값 보정
    if "answer" not in result:
        result["answer"] = raw_output

    if "health_memory_update" not in result:
        result["health_memory_update"] = {
            "should_save": False,
            "category": "none",
            "summary": "",
            "detail": {
                "keywords": [],
                "time_expr": "",
                "emotion": "",
                "need_follow_up": False,
                "risk_level": "none"
            },
            "status": "unknown"
        }

    return result


In [136]:
def chat_with_llm(user_id, user_query, k=5, verbose=True):
    """사투리 RAG + 최근 대화 메모리 + 건강 이슈 메모리를 사용하는 최종 챗봇 함수."""

    # 사용자별 건강 메모리 인스턴스 가져오기 또는 생성
    if user_id not in health_memories_by_user:
        health_memories_by_user[user_id] = MultiUserHealthMemory(user_id)
    user_health_memory = health_memories_by_user[user_id]

    # 사용자별 채팅 기록 인스턴스 가져오기 또는 생성
    if user_id not in chat_histories_by_user:
        chat_histories_by_user[user_id] = InMemoryChatMessageHistory(
            k=5, return_messages=True, memory_key="chat_history", input_key="user_input"
        )
    user_chat_history = chat_histories_by_user[user_id]

    # 1. 사투리 참고자료 검색
    dialect_context = format_dialect_context(user_query, k=k)

    # 2. 관련 건강 이슈 메모리 검색
    related_health_memories = user_health_memory.retrieve_related(user_query, max_results=5)
    health_memory_context = user_health_memory.format_for_prompt(related_health_memories)

    # 3. 최근 대화 가져오기
    recent_chat_history = format_recent_chat_history(user_chat_history)

    # 4. 사용자 정보 및 기존 건강 상태 가져오기
    user_info = users_df[users_df['user_id'] == user_id]
    user_name = user_info['name'].iloc[0] if not user_info.empty else '어르신'

    medication_rows = get_medication(user_id)
    if medication_rows:
        latest_medication = medication_rows[0]
        medication_summary = latest_medication["medicine"]
    else:
        medication_summary = "복약 기록 없음"

    latest_condition = get_latest_condition(user_id)

    if latest_condition:
        user_conditions = (
            f"최근 기록: "
            f"식사- {latest_condition['meal']}, "
            f"수면- {latest_condition['sleep']}, "
            f"기분- {latest_condition['mood']}, "
            f"통증- {latest_condition['pain']}, "
            f"복약- {medication_summary}."
        )
    else:
        user_conditions = (
            f"컨디션 기록 없음, 복약- {medication_summary}."
        )

    # 5. 프롬프트 구성
    user_prompt = build_user_prompt(
        user_input=user_query,
        dialect_context=dialect_context,
        user_name=user_name,
        user_conditions=user_conditions,
        health_memory_context=health_memory_context,
        recent_chat_history=recent_chat_history,
    )

    # 6. LLM 호출
    response = llm.invoke([
        ("system", SYSTEM_PROMPT),
        ("human", user_prompt),
    ])

    raw_output = response.content

    # 7. JSON 파싱
    result = parse_llm_json(raw_output)
    answer = result.get("answer", "")
    health_update = result.get("health_memory_update", {})

    # 8. 대화 내역 DB에 저장
    save_conversation_to_db(user_id, 'user', user_query)
    save_conversation_to_db(user_id, 'assistant', answer)

    # 9. 최근 대화 메모리 저장
    user_chat_history.add_user_message(user_query)
    user_chat_history.add_ai_message(answer)

    # 10. 건강 이슈 메모리 저장
    if health_update.get("should_save"):
        user_health_memory.add_memory(
            user_input=user_query,
            assistant_answer=answer,
            memory_update=health_update,
        )

    if verbose:
        print("\n[사용자 ID]")
        print(user_id)
        print("\n[사용자 입력]")
        print(user_query)
        print(f"\n[사용자 이름: {user_name}, 기존 건강 상태: {user_conditions}]")
        print("\n[LLM 답변]")
        print(answer)

    return answer

print("최종 chat_with_llm 함수 개인화 및 DB 저장 기능 준비 완료")

최종 chat_with_llm 함수 개인화 및 DB 저장 기능 준비 완료


이제 `HealthMemory`와 `InMemoryChatMessageHistory`를 `user_id`별로 관리하도록 수정하겠습니다. 이렇게 하면 각 사용자마다 독립적인 건강 이슈 기록과 대화 기록을 유지할 수 있습니다.

In [137]:
# user_id별로 HealthMemory 인스턴스를 관리하기 위한 딕셔너리
health_memories_by_user = {}

# user_id별로 InMemoryChatMessageHistory 인스턴스를 관리하기 위한 딕셔너리
chat_histories_by_user = {}

class MultiUserHealthMemory:
    """여러 사용자 각각의 건강 이슈 중심 장기 메모리."""

    def __init__(self, user_id):
        self.user_id = user_id
        self.memories = []

    def add_memory(self, user_input, assistant_answer, memory_update):
        self.memories.append({
            "created_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
            "category": memory_update.get("category", "none"),
            "summary": memory_update.get("summary", ""),
            "detail": memory_update.get("detail", {}),
            "status": memory_update.get("status", "unknown"),
            "original_input": user_input,
            "assistant_answer": assistant_answer,
        })

    def retrieve_related(self, user_input, max_results=5):
        """현재 입력과 관련 있는 건강 이슈 메모리를 단순 점수 기반으로 검색한다."""
        related = []

        for memory_item in self.memories:
            score = 0

            summary = memory_item.get("summary", "")
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            category = memory_item.get("category", "")
            status = memory_item.get("status", "")

            # 1. LLM이 뽑은 핵심 키워드가 현재 입력에 다시 등장하면 강하게 연결
            for keyword in keywords:
                keyword = str(keyword).strip()
                if keyword and keyword in user_input:
                    score += 3

            # 2. summary 안의 의미 있는 토큰이 현재 입력에 포함되면 약하게 연결
            for token in summary.split():
                token = token.strip(".,!?~…은는이가을를에의도만고")
                if len(token) >= 2 and token in user_input:
                    score += 1

            # 3. 이어지는 표현이면 active 이슈와 연결 가능성을 높임
            if any(word in user_input for word in ["아직도", "계속", "오늘도", "또", "그때처럼", "여전히"]):
                if status == "active":
                    score += 2

            # 4. category 자체가 입력에 직접 들어오는 경우는 드물지만 보조 점수로 둠
            if category and category != "none" and category in user_input:
                score += 1

            if score > 0:
                related.append((score, memory_item))

        related = sorted(related, key=lambda x: x[0], reverse=True)
        return [item for score, item in related[:max_results]]

    def format_for_prompt(self, related_memories):
        if not related_memories:
            return "이전 건강 이슈 메모리 없음"

        lines = []
        for i, memory_item in enumerate(related_memories, start=1):
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            lines.append(
                f"{i}. 날짜: {memory_item['created_at']} / "
                f"분류: {memory_item['category']} / "
                f"요약: {memory_item['summary']} / "
                f"키워드: {', '.join(map(str, keywords))} / "
                f"상태: {memory_item['status']}"
            )
        return "\n".join(lines)

    def to_dataframe(self):
        return pd.DataFrame(self.memories)

print("멀티 사용자 건강 이슈 메모리 클래스 준비 완료")

멀티 사용자 건강 이슈 메모리 클래스 준비 완료


# 2. 복지사 앱

복지사 앱의 이름은 `worker_*`, 앱 객체는 `worker_demo`, 라우터는 `worker_page_router`로 분리했습니다.


In [138]:

# 단디봄 복지사 앱 프론트엔드 MVP
# 환경: Google Colab + Gradio 6.5.1
# 범위: 더미 로그인 / 대시보드 / 이상 신호 관리 / 대상자 목록 / 대상자 상세

import gradio as gr
from pathlib import Path

WORKER_LOGO_PATH = Path("/content/assets/dandibom_logo.png")

if not WORKER_LOGO_PATH.exists():
    raise FileNotFoundError(f"로고 파일이 없습니다: {WORKER_LOGO_PATH}")

WORKER_LOGO_URL = f"/gradio_api/file={WORKER_LOGO_PATH}"

WORKER_COLORS = {
    "coral": "#F29B91",
    "coral_dark": "#D97D73",
    "brown": "#6B4634",
    "brown_dark": "#483126",
    "surface": "#FFF8F6",
    "border": "#E8DEDA",
    "text": "#2F2927",
    "muted": "#756D69",
    "danger": "#D9534F",
    "warning": "#E6A23C",
    "success": "#38A169",
}

WORKER_CSS = r'''
:root {
  --coral: #F29B91;
  --coral-dark: #D97D73;
  --brown: #6B4634;
  --brown-dark: #483126;
  --surface: #FFF8F6;
  --border: #E8DEDA;
  --text: #2F2927;
  --muted: #756D69;
  --danger: #D9534F;
  --warning: #E6A23C;
  --success: #38A169;
}

html, body { margin: 0 !important; background: #F4F1EF !important; overflow: hidden !important; }
.gradio-container {
  width: 100vw !important;
  max-width: 1440px !important;
  height: 100vh !important;
  max-height: 900px !important;
  margin: 0 auto !important;
  padding: 0 !important;
  overflow: hidden !important;
  background: #fff !important;
  color: var(--text) !important;
  font-family: "Noto Sans KR", "Apple SD Gothic Neo", sans-serif !important;
  box-shadow: 0 0 30px rgba(72, 49, 38, .08);
}
.app-frame { width:100% !important; height: 100% !important; gap: 0 !important; overflow: hidden !important; }
.app-frame > .form { height: 100% !important; }
.sidebar {
  width: 244px !important; min-width: 244px !important; max-width: 244px !important;
  height: 100% !important; padding: 26px 18px !important; gap: 10px !important;
  background: #FFF8F6 !important; border-right: 1px solid var(--border) !important;
  box-sizing: border-box !important;
}
.content-shell { height: 100% !important; min-width:0 !important; padding: 26px 30px !important; gap: 16px !important; overflow: hidden !important; }
.content-shell .row, .content-shell .column, .content-shell .panel, .content-shell .block { min-width:0 !important; }
.brand img { width: 150px; height: 104px; object-fit: contain; display: block; margin: 0 auto 18px; }
.brand-mini { display:flex; align-items:center; gap:12px; }
.brand-mini img { width: 58px; height: 52px; object-fit: contain; }
.brand-mini strong { color:var(--brown); font-size:1.2rem; }
.worker-chip { padding: 9px 12px; border:1px solid var(--border); border-radius:12px; background:#fff; font-size:.92rem; color:var(--brown); }
.nav-title { margin: 18px 8px 4px; color: var(--muted); font-size: .78rem; font-weight: 800; letter-spacing: .08em; }
.nav-btn, .nav-btn button {
  width:100% !important; min-height:48px !important; border-radius:12px !important;
  justify-content:flex-start !important; padding:0 15px !important;
  background:transparent !important; border:1px solid transparent !important;
  color:var(--brown-dark) !important; font-weight:750 !important; box-shadow:none !important;
}
.nav-btn:hover, .nav-btn button:hover { background:#fff !important; border-color:var(--border) !important; }
.nav-active, .nav-active button { background:#FCE2DD !important; border-color:#F4C3BA !important; color:var(--brown) !important; }
.logout-btn, .logout-btn button { margin-top:auto !important; background:#fff !important; border:1px solid var(--border) !important; color:var(--muted) !important; }
.page-header { display:flex; align-items:center; justify-content:space-between; min-height:58px; padding-bottom:14px; border-bottom:1px solid var(--border); }
.page-header h1 { margin:0; font-size:1.65rem; color:var(--brown-dark); }
.page-header p { margin:5px 0 0; color:var(--muted); font-size:.93rem; }
.header-badge { padding:8px 12px; border:1px solid var(--border); border-radius:10px; background:#fff; color:var(--brown); font-weight:700; }
.worker_metric-card { border:1px solid var(--border) !important; border-radius:16px !important; background:#fff !important; padding:17px 19px !important; box-shadow:0 4px 14px rgba(72,49,38,.05); }
.worker_metric-label { color:var(--muted); font-size:.9rem; margin-bottom:8px; }
.worker_metric-value { color:var(--brown-dark); font-size:1.65rem; font-weight:900; }
.worker_metric-danger { color:var(--danger); }
.worker_metric-warning { color:#B87913; }
.worker_metric-success { color:var(--success); }
.panel { border:1px solid var(--border) !important; border-radius:16px !important; background:#fff !important; padding:18px !important; overflow:hidden !important; }
.panel-title { margin:0 0 12px; color:var(--brown-dark); font-size:1.08rem; font-weight:850; }
.target-btn, .target-btn button { width:100% !important; max-width:100% !important; min-width:0 !important; min-height:76px !important; text-align:left !important; justify-content:flex-start !important; white-space:pre-line !important; overflow-wrap:anywhere !important; overflow:hidden !important; border-radius:12px !important; background:#fff !important; border:1px solid var(--border) !important; color:var(--text) !important; }
.target-danger, .target-danger button { background:#FFF3DD !important; border-color:#F0CF8E !important; }
.alert-summary { border:1px solid #F0CF8E; border-radius:14px; background:#FFF5DF; padding:18px; }
.alert-summary .quote { color:#A85B00; font-size:1.35rem; font-weight:900; margin:8px 0; }
.status-dot { display:inline-block; width:11px; height:11px; border-radius:50%; margin-right:6px; }
.status-red { background:var(--danger); }
.status-yellow { background:var(--warning); }
.status-green { background:var(--success); }
.info-grid { display:grid; grid-template-columns:1fr 1fr; gap:14px; }
.info-box { border:1px solid var(--border); border-radius:12px; padding:14px; background:#fff; }
.info-box small { color:var(--muted); display:block; margin-bottom:7px; }
.info-box strong { color:var(--brown-dark); font-size:1.05rem; }
.primary-btn, .primary-btn button { background:var(--coral) !important; border:none !important; color:#fff !important; font-weight:850 !important; border-radius:11px !important; }
.secondary-btn, .secondary-btn button { background:#fff !important; border:1px solid var(--border) !important; color:var(--brown) !important; font-weight:750 !important; border-radius:11px !important; }
.danger-outline, .danger-outline button { background:#fff !important; border:1px solid #E6A29F !important; color:#B74440 !important; font-weight:800 !important; border-radius:11px !important; }
.filter-btn, .filter-btn button { min-height:50px !important; justify-content:flex-start !important; border-radius:11px !important; background:#fff !important; border:1px solid var(--border) !important; color:var(--brown-dark) !important; }
.filter-active, .filter-active button { background:#FCE2DD !important; border-color:#F3BBB0 !important; }
.alert-row, .alert-row button { min-height:70px !important; justify-content:flex-start !important; text-align:left !important; white-space:pre-line !important; border-radius:12px !important; background:#F8F7F6 !important; border:1px solid transparent !important; color:var(--text) !important; }
.alert-unread, .alert-unread button { background:#FFF3DD !important; border-color:#F0CF8E !important; color:#8A4B00 !important; }
.alert-line { position:relative !important; width:100% !important; min-width:0 !important; gap:8px !important; align-items:flex-start !important; }
.alert-line > .form:first-child { min-width:0 !important; flex:1 1 auto !important; }
.alert-confirm, .alert-confirm button {
  width:72px !important; min-width:72px !important; max-width:72px !important;
  min-height:38px !important; margin-top:8px !important; border-radius:10px !important;
  background:var(--coral) !important; border:none !important; color:#fff !important;
  font-weight:850 !important; padding:0 10px !important;
}
.table-head { display:grid; grid-template-columns:1.3fr .5fr 1fr 1.5fr .7fr; gap:12px; padding:11px 15px; color:var(--muted); font-size:.82rem; font-weight:800; border-bottom:1px solid var(--border); }
.subject-row, .subject-row button { min-height:58px !important; justify-content:flex-start !important; text-align:left !important; font-family:inherit !important; border-radius:9px !important; background:#fff !important; border:1px solid var(--border) !important; color:var(--text) !important; }
.subject-row button:hover, .subject-row:hover { background:var(--surface) !important; border-color:#F2BEB5 !important; }
.detail-hero { border:1px solid var(--border); border-radius:16px; padding:22px; background:linear-gradient(135deg,#FFF8F6,#fff); }
.detail-hero h2 { margin:0 0 7px; color:var(--brown-dark); font-size:1.5rem; }
.detail-hero p { margin:0; color:var(--muted); }
.feature-btn, .feature-btn button { height:130px !important; border-radius:16px !important; background:#fff !important; border:1px solid var(--border) !important; color:var(--brown-dark) !important; font-size:1.08rem !important; font-weight:850 !important; box-shadow:0 4px 14px rgba(72,49,38,.05); }
.login-stage { height:100% !important; align-items:center !important; justify-content:center !important; background:#FFFDFC !important; }
.login-card { width:520px !important; min-width:520px !important; max-width:520px !important; padding:34px 48px !important; border:1px solid var(--border) !important; border-radius:22px !important; background:#fff !important; box-shadow:0 12px 40px rgba(72,49,38,.09); }
.login-card .brand img { width:175px; height:130px; margin-bottom:8px; }
.login-title { text-align:center; margin:0; color:var(--brown-dark); font-size:1.5rem; }
.login-sub { text-align:center; margin:7px 0 20px; color:var(--muted); }
.login-input input { min-height:48px !important; font-size:1rem !important; }
.login-error { color:var(--danger); text-align:center; min-height:22px; }
footer { display:none !important; }
'''

WORKER_TARGETS = [
    {"name": "김순자", "age": 76, "guardian": "이정미", "state": "위험 신호 미확인", "time": "오늘 10:07"},
    {"name": "박말례", "age": 81, "guardian": "박진우", "state": "오늘 대화 완료 · 양호", "time": "오늘 09:42"},
    {"name": "이팔복", "age": 79, "guardian": "이영희", "state": "오늘 대화 완료 · 양호", "time": "오늘 09:18"},
    {"name": "최복순", "age": 84, "guardian": "최민수", "state": "아직 대화 없음", "time": "-"},
    {"name": "한정자", "age": 78, "guardian": "한서윤", "state": "복약 확인 필요", "time": "어제 18:30"},
]

WORKER_ALERTS = [
    {"name": "김순자", "content": '"요 며칠 어지럽고 기운이 없어예"', "time": "오늘 10:07", "status": "미확인", "level": "위험"},
    {"name": "이팔복", "content": "수면 불규칙 감지", "time": "7/12", "status": "확인 완료", "level": "주의"},
    {"name": "박말례", "content": '"입맛이 영 없다" 반복 언급', "time": "7/10", "status": "확인 완료", "level": "주의"},
]


def worker_logo_html():
    return f'<div class="brand"><img src="{WORKER_LOGO_URL}" alt="단디봄 로고"></div>'


def go(target):
    return target


def worker_verify_login(login_id, password):
    if not (login_id or "").strip() or not (password or "").strip():
        return "login", "아이디와 비밀번호를 입력해주세요."
    return "dashboard", ""


def worker_render_header(title, subtitle):
    gr.HTML(f'''
    <div class="page-header">
      <div><h1>{title}</h1><p>{subtitle}</p></div>
      <div class="header-badge">이경미 복지사님</div>
    </div>
    ''')


def worker_nav_button(label, target, current, key):
    classes = ["nav-btn"]
    if target == current:
        classes.append("nav-active")
    btn = gr.Button(label, elem_classes=classes, key=key)
    btn.click(lambda t=target: t, outputs=worker_page_router, queue=False, key=f"{key}-event")
    return btn


def worker_render_sidebar(current):
    with gr.Column(elem_classes="sidebar", scale=0):
        gr.HTML(f'<div class="brand-mini"><img src="{WORKER_LOGO_URL}"><strong>단디봄</strong></div>')
        gr.HTML('<div class="worker-chip">복지사 · 이경미</div>')
        gr.HTML('<div class="nav-title">업무 메뉴</div>')
        worker_nav_button("▦  대시보드", "dashboard", current, f"nav-dashboard-{current}")
        worker_nav_button("⚠  이상 신호 관리", "alerts", current, f"nav-alerts-{current}")
        worker_nav_button("♙  대상자 목록", "subjects", current, f"nav-subjects-{current}")
        gr.HTML('<div class="nav-title">계정</div>')
        logout = gr.Button("↪  로그아웃", elem_classes=["nav-btn", "logout-btn"], key=f"logout-{current}")
        logout.click(lambda: "login", outputs=worker_page_router, queue=False, key=f"logout-{current}-event")


def worker_metric(label, value, tone=""):
    tone_class = f" worker_metric-{tone}" if tone else ""
    gr.HTML(f'<div class="worker_metric-card"><div class="worker_metric-label">{label}</div><div class="worker_metric-value{tone_class}">{value}</div></div>')


WORKER_AGE_BY_USER_ID = {1: 76, 4: 81, 5: 79, 6: 84}

def worker_db_summary():
    with get_db_connection() as conn:
        target_count = conn.execute("SELECT COUNT(*) FROM users WHERE role='대상자'").fetchone()[0]
        pending_count = conn.execute("SELECT COUNT(*) FROM alerts WHERE status='대기'").fetchone()[0]
        confirmed_count = conn.execute("SELECT COUNT(*) FROM alerts WHERE status='확인'").fetchone()[0]
    return target_count, pending_count, confirmed_count

def worker_db_alerts():
    with get_db_connection() as conn:
        return conn.execute("""
            SELECT a.alert_id, a.user_id, u.name, a.cause, a.level, a.status, a.created_at
            FROM alerts a JOIN users u ON u.user_id = a.user_id
            ORDER BY CASE WHEN a.status='대기' THEN 0 ELSE 1 END, a.created_at DESC
        """).fetchall()

def worker_db_targets():
    with get_db_connection() as conn:
        return conn.execute("""
            SELECT u.user_id, u.name, g.name AS guardian,
                   MAX(a.created_at) AS latest_alert,
                   SUM(CASE WHEN a.status='대기' THEN 1 ELSE 0 END) AS pending_alerts
            FROM relationships r
            JOIN users u ON u.user_id=r.target_id
            LEFT JOIN users g ON g.user_id=r.guardian_id
            LEFT JOIN alerts a ON a.user_id=u.user_id
            WHERE r.worker_id=3
            GROUP BY u.user_id, u.name, g.name
            ORDER BY u.user_id
        """).fetchall()

def worker_confirm_alert_one(refresh_value=0):
    """시연 기준 위험 신호 alert_id=1을 확인 처리하고 화면 갱신값을 증가시킵니다."""
    with get_db_connection() as conn:
        conn.execute(
            "UPDATE alerts SET status='확인' WHERE alert_id=1 AND status='대기'"
        )
        conn.commit()
    return int(refresh_value or 0) + 1

def worker_db_subject_detail(user_id=1):
    with get_db_connection() as conn:
        return conn.execute("""
            SELECT u.user_id, u.name, u.code, g.name AS guardian, w.name AS worker
            FROM relationships r
            JOIN users u ON u.user_id=r.target_id
            LEFT JOIN users g ON g.user_id=r.guardian_id
            LEFT JOIN users w ON w.user_id=r.worker_id
            WHERE u.user_id=?
        """, (user_id,)).fetchone()

def worker_render_dashboard():
    worker_render_header("복지사 대시보드", "담당 대상자의 오늘 상태와 위험 신호를 확인합니다.")
    target_count, pending_count, confirmed_count = worker_db_summary()
    alerts_data = worker_db_alerts()
    top_alert = alerts_data[0] if alerts_data else None
    with gr.Row(equal_height=True):
        with gr.Column(): worker_metric("담당 대상자", f"{target_count}명")
        with gr.Column(): worker_metric("미확인 위험 신호", f"{pending_count}건", "danger")
        with gr.Column(): worker_metric("확인 완료", f"{confirmed_count}건", "success")
        with gr.Column(): worker_metric("오늘 대화 미참여", "1명", "warning")

    with gr.Row(equal_height=True):
        with gr.Column(scale=4, elem_classes="panel"):
            gr.HTML('<h2 class="panel-title">담당 대상자</h2>')
            for idx, target in enumerate(worker_db_targets()[:3]):
                age = WORKER_AGE_BY_USER_ID.get(target["user_id"], "-")
                danger = bool(target["pending_alerts"])
                icon = "🚨" if danger else "●"
                state = "위험 신호 미확인" if danger else "확인 완료"
                btn = gr.Button(
                    f"{icon} {target['name']} ({age})\n{state}",
                    elem_classes=["target-btn"] + (["target-danger"] if danger else []),
                    key=f"dash-target-{idx}",
                )
                btn.click(lambda: "subject_detail", outputs=worker_page_router, queue=False)
            more = gr.Button("+ 전체 대상자 보기", elem_classes="secondary-btn", key="dash-more")
            more.click(lambda: "subjects", outputs=worker_page_router, queue=False)

        with gr.Column(scale=8, elem_classes="panel"):
            if top_alert:
                age = WORKER_AGE_BY_USER_ID.get(top_alert["user_id"], "-")
                gr.HTML(f'<h2 class="panel-title">{top_alert["name"]} ({age}) 어르신 위험 신호</h2>')
                gr.HTML(f"""<div class="alert-summary"><strong>⚠ 이상 신호 · {top_alert["created_at"]}</strong>
                <div class="quote">{top_alert["cause"]}</div><div>상태: {top_alert["status"]} · 단계: {top_alert["level"]}</div></div>""")
            else:
                gr.HTML('<div class="info-box"><strong>등록된 이상 신호가 없습니다.</strong></div>')
            with gr.Row():
                alerts = gr.Button("이상 신호 확인", elem_classes="danger-outline", key="dash-alert")
                alerts.click(lambda: "alerts", outputs=worker_page_router, queue=False)
                detail = gr.Button("대상자 상세", elem_classes="secondary-btn", key="dash-detail")
                detail.click(lambda: "subject_detail", outputs=worker_page_router, queue=False)

def worker_render_alerts():
    worker_render_header("이상 신호 관리", "DB에 저장된 위험 발화와 건강 이상 신호를 확인합니다.")
    rows = worker_db_alerts()
    pending = sum(row["status"] == "대기" for row in rows)
    with gr.Row(equal_height=True):
        with gr.Column(scale=3, elem_classes="panel"):
            gr.HTML('<h2 class="panel-title">상태</h2>')
            gr.Button(f"전체 ({len(rows)}건)", elem_classes=["filter-btn", "filter-active"])
            gr.Button(f"미확인 ({pending}건)", elem_classes="filter-btn")
            gr.Button(f"확인 완료 ({len(rows)-pending}건)", elem_classes="filter-btn")
        with gr.Column(scale=9, elem_classes="panel"):
            gr.HTML('<div style="font-weight:800;padding:8px 12px">대상자 · 감지 내용 · 시각 · 상태</div>')
            for idx, row in enumerate(rows):
                unread = row["status"] == "대기"
                with gr.Row(elem_classes="alert-line"):
                    signal = gr.Button(
                        f'{"🚨" if unread else "✓"} {row["name"]}     {row["cause"]}     {row["created_at"]}     {"미확인" if unread else "완료"}',
                        elem_classes=["alert-row"] + (["alert-unread"] if unread else []),
                        key=f"db-alert-{idx}",
                        scale=1,
                    )
                    signal.click(lambda: "subject_detail", outputs=worker_page_router, queue=False)

                    # alert_id=1이 미확인일 때만 행 오른쪽 위에 확인 버튼을 표시합니다.
                    if row["alert_id"] == 1 and unread:
                        confirm = gr.Button(
                            "확인",
                            elem_classes="alert-confirm",
                            key="confirm-alert-1",
                            scale=0,
                            min_width=72,
                        )
                        confirm.click(
                            fn=worker_confirm_alert_one,
                            inputs=worker_alert_refresh,
                            outputs=worker_alert_refresh,
                            queue=False,
                            key="confirm-alert-1-event",
                        )
            back = gr.Button("← 대시보드", elem_classes="secondary-btn", key="alert-back")
            back.click(lambda: "dashboard", outputs=worker_page_router, queue=False)

def worker_render_subjects():
    worker_render_header("대상자 목록", "DB의 담당 대상자를 확인하고 상세 정보로 이동합니다.")
    with gr.Column(elem_classes="panel"):
        with gr.Row():
            gr.Textbox(placeholder="이름 또는 보호자명 검색 (프로토타입에서는 작동하지 않음)", label="대상자 검색", scale=8)
            gr.Button("검색", elem_classes="secondary-btn", scale=1)
        gr.HTML('<div class="table-head"><span>대상자</span><span>나이</span><span>보호자</span><span>최근 상태</span><span>확인 시각</span></div>')
        for idx, target in enumerate(worker_db_targets()):
            age = WORKER_AGE_BY_USER_ID.get(target["user_id"], "-")
            guardian = target["guardian"] or "미등록"
            danger = bool(target["pending_alerts"])
            state = "위험 신호 미확인" if danger else "확인 완료"
            latest = target["latest_alert"] or "-"
            btn = gr.Button(
                f'{"🚨" if danger else "●"} {target["name"]} | {age}세 | {guardian} | {state} | {latest}',
                elem_classes="subject-row", key=f"db-subject-{idx}",
            )
            btn.click(lambda: "subject_detail", outputs=worker_page_router, queue=False)

def worker_render_subject_detail():
    worker_render_header("대상자 상세", "대상자의 기록과 인증 정보를 관리합니다.")
    target = worker_db_subject_detail(1)
    name = target["name"] if target else "김순자"
    guardian = (target["guardian"] or "미등록") if target else "미등록"
    worker = (target["worker"] or "미등록") if target else "미등록"
    code = target["code"] if target else "-"
    age = WORKER_AGE_BY_USER_ID.get(1, "-")
    gr.HTML(f"""<div class="detail-hero"><h2>{name} 어르신 <span style="font-size:1rem;color:#756D69">({age}세)</span></h2>
    <p>보호자: {guardian} · 담당 복지사: {worker} · 인증 코드: {code}</p></div>""")
    with gr.Row(equal_height=True):
        gr.Button("▤\n건강 기록 조회", elem_classes="feature-btn", key="detail-health")
        gr.Button("AI 건강 요약", elem_classes="feature-btn", key="detail-ai")
    with gr.Row(equal_height=True):
        gr.Button("✓\n대응 기록", elem_classes="feature-btn", key="detail-response")
        gr.Button("🔑\n인증정보", elem_classes="feature-btn", key="detail-auth")
    with gr.Row():
        alerts = gr.Button("이 대상자의 이상 신호 보기", elem_classes="danger-outline", key="detail-alerts")
        alerts.click(lambda: "alerts", outputs=worker_page_router, queue=False)
        subjects = gr.Button("← 대상자 목록", elem_classes="secondary-btn", key="detail-subjects")
        subjects.click(lambda: "subjects", outputs=worker_page_router, queue=False)


with gr.Blocks(title="단디봄 복지사 앱") as worker_demo:
    worker_page_router = gr.Textbox(value="login", visible=False, key="worker-page-router")
    # 확인 처리 후 이 값만 변경하여 현재 화면을 DB 기준으로 다시 그립니다.
    worker_alert_refresh = gr.Number(value=0, visible=False, key="worker-alert-refresh")

    @gr.render(inputs=[worker_page_router, worker_alert_refresh])
    def render_worker_page(page, _alert_refresh):
        page = (page or "login").strip()
        if page == "login":
            with gr.Row(elem_classes="login-stage"):
                with gr.Column(elem_classes="login-card"):
                    gr.HTML(worker_logo_html())
                    gr.HTML('<h1 class="login-title">복지사 로그인</h1><p class="login-sub">담당 대상자 관리 화면으로 이동합니다.</p>')
                    login_id = gr.Textbox(label="이메일 또는 아이디", placeholder="worker@dandibom.kr", elem_classes="login-input", key="worker-id")
                    password = gr.Textbox(label="비밀번호", placeholder="비밀번호를 입력해주세요", type="password", elem_classes="login-input", key="worker-password")
                    login_error = gr.Markdown("", elem_classes="login-error", key="worker-login-error")
                    login_btn = gr.Button("로그인", elem_classes="primary-btn", key="worker-login-button")
                    login_btn.click(worker_verify_login, inputs=[login_id, password], outputs=[worker_page_router, login_error], queue=False, key="worker-login-event")
                    password.submit(worker_verify_login, inputs=[login_id, password], outputs=[worker_page_router, login_error], queue=False, key="worker-login-submit")
        else:
            with gr.Row(elem_classes="app-frame"):
                worker_render_sidebar(page)
                with gr.Column(elem_classes="content-shell"):
                    if page == "dashboard":
                        worker_render_dashboard()
                    elif page == "alerts":
                        worker_render_alerts()
                    elif page == "subjects":
                        worker_render_subjects()
                    elif page == "subject_detail":
                        worker_render_subject_detail()
                    else:
                        gr.Markdown("화면을 찾을 수 없습니다.")
                        reset = gr.Button("대시보드로", elem_classes="primary-btn")
                        reset.click(lambda: "dashboard", outputs=worker_page_router, queue=False)


## 사용자 앱 (ver3.2 · 위험 발화 알림)

시연 권장 대사:

> 요 며칠 자꾸 어지럽고 입맛도 하나도 없어서 밥을 못 먹겠어예.

동작:
- `어지럼` 계열 표현과 `입맛/식욕저하` 계열 표현이 한 발화에 함께 있으면 감지합니다.
- 새 알림을 추가하지 않고 DB의 `alert_id=1`을 `대기` 상태로 맞춥니다.
- LLM 답변 아래에 복지사 알림 피드백을 표시하고 Gradio 경고 메시지를 띄웁니다.


In [139]:

# 단디봄 사용자 앱 프론트엔드 MVP
# 환경: Google Colab + Gradio
# 화면 전환: visible=True/False가 아닌 gr.render 기반
# 백엔드/DB 연결 전 단계이므로 UserBackendAPI의 더미 함수만 교체하면 됩니다.

import gradio as gr

USER_LOGO_URL = "/gradio_api/file=/content/assets/dandibom_logo.png"

PRIMARY = "#F29B91"
PRIMARY_DARK = "#D97D73"
SECONDARY = "#B8BD70"
BROWN = "#6B4634"
BACKGROUND = "#FFFFFF"
SURFACE = "#FFF9F7"
BORDER = "#E8DEDA"
TEXT = "#2F2927"
MUTED = "#756D69"
WARNING = "#F4B24A"
DANGER = "#D95C5C"
SUCCESS = "#6E9F67"

USER_CSS = f"""
:root {{
  --primary: {PRIMARY};
  --primary-dark: {PRIMARY_DARK};
  --secondary: {SECONDARY};
  --brown: {BROWN};
  --background: {BACKGROUND};
  --surface: {SURFACE};
  --border: {BORDER};
  --text: {TEXT};
  --muted: {MUTED};
  --warning: {WARNING};
  --danger: {DANGER};
  --success: {SUCCESS};
}}

.gradio-container {{
  background: var(--background) !important;
  color: var(--text) !important;
  width: 100% !important;
  max-width: 430px !important;
  min-width: 320px !important;
  margin: 0 auto !important;
  min-height: 100vh;
  padding-bottom: 0 !important;
  box-sizing: border-box !important;
  font-family: "Noto Sans KR", "Apple SD Gothic Neo", sans-serif;
}}

.app-shell {{ padding: 12px 6px 24px; }}

.brand-logo img {{
  width: 132px;
  max-height: 132px;
  object-fit: contain;
  display: block;
  margin: 0 auto 8px;
}}

.page-title {{
  font-size: 1.55rem;
  font-weight: 800;
  color: var(--brown);
  margin: 0;
}}

.page-subtitle {{
  color: var(--muted);
  font-size: 1.05rem;
  line-height: 1.55;
}}

.card {{
  border: 1px solid var(--border);
  border-radius: 18px;
  background: #fff;
  padding: 18px;
  box-shadow: 0 3px 14px rgba(93, 63, 48, 0.06);
}}

.soft-card {{
  border: 1px solid #F0DED9;
  border-radius: 18px;
  background: var(--surface);
  padding: 18px;
}}

.status-ok {{ color: var(--success); font-weight: 800; }}
.status-warn {{ color: #A56710; font-weight: 800; }}

.header-row {{
  width: 100% !important;
  align-items: center;
  border-bottom: 1px solid var(--border);
  padding-bottom: 10px;
  margin-bottom: 16px;
}}

button.header-back {{
  min-width: 32px !important;
  width: 32px !important;
  height: 40px !important;
  flex: 0 0 60px !important;
}}

.menu-button button {{
  min-height: 66px !important;
  border-radius: 17px !important;
  justify-content: flex-start !important;
  padding: 0 20px !important;
  background: #fff !important;
  border: 1px solid var(--border) !important;
  color: var(--text) !important;
  font-size: 1.15rem !important;
  font-weight: 750 !important;
  box-shadow: 0 2px 10px rgba(93, 63, 48, 0.05);
}}

.menu-button button:hover {{
  border-color: var(--primary) !important;
  background: var(--surface) !important;
}}

.primary-btn button {{
  min-height: 58px !important;
  border-radius: 17px !important;
  background: var(--primary) !important;
  border: none !important;
  color: #fff !important;
  font-size: 1.15rem !important;
  font-weight: 800 !important;
}}

.primary-btn button:hover {{ background: var(--primary-dark) !important; }}

.secondary-btn button {{
  min-height: 54px !important;
  border-radius: 17px !important;
  background: #fff !important;
  border: 1px solid var(--border) !important;
  color: var(--text) !important;
  font-size: 1.08rem !important;
  font-weight: 750 !important;
}}

.danger-btn button {{
  min-height: 60px !important;
  border-radius: 17px !important;
  background: var(--warning) !important;
  border: none !important;
  color: #4A3216 !important;
  font-size: 1.2rem !important;
  font-weight: 900 !important;
}}


.big-input textarea,
.big-input input {{
  font-size: 1.12rem !important;
  min-height: 54px !important;
}}

.gr-checkbox label,
.gr-radio label {{
  font-size: 1.08rem !important;
  line-height: 1.5 !important;
}}


/* ---- Fixed home navigation with reserved content space ---- */
.gradio-container {{
  width: 100% !important;
  max-width: 430px !important;
  min-width: 320px !important;
  margin: 0 auto !important;
  min-height: 100vh !important;
  box-sizing: border-box !important;
}}

.app-shell {{
  width: 100% !important;
  max-width: 430px !important;
  margin: 0 auto !important;
  padding: 12px 16px 122px !important;
  box-sizing: border-box !important;
}}

.home-nav-space {{
  position: fixed !important;
  left: 50% !important;
  bottom: 0 !important;
  transform: translateX(-50%) !important;
  width: 100% !important;
  max-width: 430px !important;
  min-height: 94px !important;
  display: flex !important;
  justify-content: flex-end !important;
  align-items: center !important;
  padding: 14px 16px calc(14px + env(safe-area-inset-bottom)) !important;
  border-top: 1px solid var(--border) !important;
  background: rgba(255, 255, 255, 0.98) !important;
  box-shadow: 0 -6px 18px rgba(93, 63, 48, 0.08) !important;
  box-sizing: border-box !important;
  z-index: 9999 !important;
}}

.home-nav-button {{
  width: 96px !important;
  min-width: 96px !important;
  max-width: 96px !important;
}}

.home-nav-button button {{
  width: 96px !important;
  min-width: 96px !important;
  max-width: 96px !important;
  height: 56px !important;
  border-radius: 18px !important;
  background: var(--secondary) !important;
  border: none !important;
  color: #fff !important;
  font-size: 1.05rem !important;
  font-weight: 900 !important;
  box-shadow: 0 5px 16px rgba(82, 86, 42, 0.18);
}}

/* ---- Galaxy S8+ (360 x 740 USER_CSS px) compact health screens ---- */
@media (max-width: 430px) and (max-height: 800px) {{
  .app-shell {{ padding: 8px 12px 98px !important; gap: 6px !important; }}
  .header-row {{ padding-bottom: 6px !important; margin-bottom: 6px !important; }}
  .header-row .page-title {{ font-size: 1.3rem !important; }}
  .home-nav-space {{ min-height: 76px !important; padding: 8px 12px !important; }}
  .home-nav-button, .home-nav-button button {{ height: 48px !important; }}
}}

.survey-card {{
  border: 1px solid var(--border);
  border-radius: 16px;
  padding: 14px 10px 12px;
  background: #fff;
  gap: 4px !important;
}}
.survey-title {{ text-align:center; font-size:1.28rem; font-weight:900; margin:0 0 4px; }}
.survey-note {{ font-size:.78rem; color:var(--muted); line-height:1.35; margin:2px 0; }}
.survey-help {{ text-align:center; font-size:.82rem; color:var(--muted); margin:2px 0 0; }}
.compact-radio {{ margin: 0 !important; padding: 0 !important; }}
.compact-radio > label {{ text-align:center !important; font-size:.96rem !important; font-weight:700 !important; }}
.compact-radio .wrap {{ gap: 6px !important; }}
.compact-radio label:has(input) {{
  min-height: 38px !important; border: 1px solid #D7D1CE !important;
  border-radius: 11px !important; justify-content: center !important;
  padding: 5px 8px !important; font-size: 1rem !important; font-weight: 750 !important;
}}
.compact-radio label:has(input:checked) {{ background:#CFE2FF !important; border-color:#AFCBF2 !important; }}
.compact-next button {{ min-height:52px !important; border-radius:12px !important; font-size:1.12rem !important; }}
.chat-compact {{ min-height: 0 !important; }}
.chat-compact .chatbot {{ min-height: 0 !important; }}

footer {{ display: none !important; }}
"""

# ---------------------------------------------------------------------
# 2. 백엔드 대체용 인터페이스
# ---------------------------------------------------------------------
class UserBackendAPI:
    @staticmethod
    def verify_code(code: str) -> dict:
        code = (code or "").strip().upper()
        if not code:
            return {"ok": False, "message": "코드를 입력해주세요."}
        with get_db_connection() as conn:
            row = conn.execute(
                "SELECT user_id, name FROM users WHERE code = ? AND role = '대상자'",
                (code,),
            ).fetchone()
        if row is None:
            return {"ok": False, "message": "인증 코드가 올바르지 않습니다."}
        return {"ok": True, "user_id": row["user_id"], "name": row["name"]}

    @staticmethod
    def get_medications(user_id: int) -> list[dict]:
        return [
            {"id": 1, "name": "혈압약", "time": "아침 식후", "taken": False},
            {"id": 2, "name": "관절약", "time": "저녁 식후", "taken": True},
        ]

    @staticmethod
    def record_medication_and_start_chat(user_id: int, morning: str) -> str:
        """아침 복약 선택을 저장한 뒤, 기록을 근거로 AI가 먼저 말을 건넵니다."""
        morning = morning or "아직요"
        save_morning_medication(user_id, morning)
        prompt = (
            f"어르신이 방금 아침약 복용 여부를 '{morning}'라고 기록했습니다. "
            "이 기록을 바탕으로 짧고 다정하게 먼저 말을 걸고, 필요한 경우 한 가지 질문만 해주세요. "
            "내부 처리나 기록 저장 사실을 설명하지 마세요."
        )
        return chat_with_llm(user_id, prompt, verbose=False)

    @staticmethod
    def record_condition_and_start_chat(
        user_id: int, meal: str, sleep: str, mood: str, pain: str
    ) -> str:
        """컨디션 선택을 저장한 뒤, 기록을 근거로 AI가 먼저 말을 건넵니다."""
        meal = meal or "아침"
        sleep = sleep or "보통"
        mood = mood or "보통"
        pain = pain or "없음"
        meal_json = json.dumps({
            "breakfast": "y" if meal == "아침" else "n",
            "lunch": "y" if meal == "점심" else "n",
            "dinner": "y" if meal == "저녁" else "n",
        }, ensure_ascii=False)
        save_condition(user_id, meal_json, sleep, mood, pain)
        prompt = (
            "어르신이 방금 컨디션을 기록했습니다. "
            f"식사={meal}, 수면={sleep}, 기분={mood}, 통증={pain}. "
            "이 기록을 바탕으로 짧고 다정하게 먼저 말을 걸고, 가장 필요한 한 가지 질문만 해주세요. "
            "내부 처리나 기록 저장 사실을 설명하지 마세요."
        )
        return chat_with_llm(user_id, prompt, verbose=False)

    @staticmethod
    def chat(user_id: int, message: str) -> str:
        text = (message or "").strip()
        if not text:
            return "천천히 말씀해주셔도 괜찮습니다."
        return chat_with_llm(user_id, text, verbose=False)

# ---------------------------------------------------------------------
# 3. 화면 이동
#    Colab + gr.render 안정성을 위해 화면 상태(user_page_router) 하나만 변경합니다.
# ---------------------------------------------------------------------
def user_go_page(target: str):
    return target

def user_chat_route(assistant_message: str) -> str:
    """페이지와 첫 AI 답변을 하나의 라우터 값으로 전달해 중복 렌더링을 막습니다."""
    return json.dumps(
        {"page": "chat", "seed": assistant_message or ""},
        ensure_ascii=False,
    )

def user_is_demo_risk_utterance(text: str) -> bool:
    """시연용: 어지럼과 식욕저하가 함께 언급되면 alert_id=1 시나리오를 실행합니다."""
    normalized = (text or "").replace(" ", "")
    dizziness_words = ("어지럽", "어지러", "빙빙", "휘청")
    appetite_words = ("입맛", "식욕", "밥을못", "밥도못", "못먹", "안먹")
    return (
        any(word in normalized for word in dizziness_words)
        and any(word in normalized for word in appetite_words)
    )

def user_activate_demo_alert(user_id: int) -> bool:
    """새 행을 만들지 않고 시연 기준 데이터 alert_id=1을 미확인 상태로 활성화합니다."""
    with get_db_connection() as conn:
        cursor = conn.execute(
            """UPDATE alerts
               SET status='대기'
               WHERE alert_id=1 AND user_id=?
                 AND cause='어지럼·식욕저하 발화 감지'""",
            (int(user_id),),
        )
        conn.commit()
        return cursor.rowcount == 1

def user_verify_and_enter(code: str):
    result = UserBackendAPI.verify_code(code)
    if result["ok"]:
        return "home", "", int(result["user_id"])
    return "code", f"⚠️ {result['message']}", 1

# ---------------------------------------------------------------------
# 4. 공통 UI
# ---------------------------------------------------------------------
def user_logo_html(compact: bool = False):
    width = 88 if compact else 132
    return f"""
    <div class="brand-logo">
      <img src="{USER_LOGO_URL}" style="width:{width}px" alt="단디봄 로고">
    </div>
    """

def user_render_header(title: str, back_target: str, current_page: str):
    with gr.Row(elem_classes="header-row"):
        back = gr.Button(
            "←",
            elem_classes="header-back",
            scale=0,
            min_width=40,
            key=f"back-{current_page}",
        )
        gr.HTML(f'<h1 class="page-title">{title}</h1>')
    back.click(
        fn=lambda target=back_target: target,
        outputs=user_page_router,
        key=f"back-event-{current_page}",
        queue=False,
    )

def user_render_home_button():
    # 하단 fixed 홈 영역의 높이만큼 app-shell에 여백을 예약하여 콘텐츠 침범을 방지합니다.
    with gr.Row(elem_classes="home-nav-space"):
        home = gr.Button(
            "⌂ 홈",
            elem_classes="home-nav-button",
            min_width=96,
            scale=0,
            key="home-nav-button",
        )
    home.click(
        fn=lambda: "home",
        outputs=user_page_router,
        key="home-event",
        queue=False,
    )

def user_menu_button(label: str, target: str, key: str):
    button = gr.Button(label, elem_classes="menu-button", key=key)
    button.click(
        fn=lambda t=target: t,
        outputs=user_page_router,
        key=f"nav-event-{key}",
        queue=False,
    )
    return button

# ---------------------------------------------------------------------
# 5. 앱
# ---------------------------------------------------------------------
with gr.Blocks(title="단디봄") as user_demo:
    # gr.State 대신 숨김 Textbox를 화면 라우터로 사용합니다.
    # Colab/공유 링크 환경에서 State 변경이 gr.render 재실행으로 이어지지 않는 경우를 피하기 위함입니다.
    user_page_router = gr.Textbox(value="code", visible=False, key="page-router")
    user_app_user_id_state = gr.State(1)

    # 라우터 하나만 gr.render 입력으로 사용합니다. 두 입력이 연속 변경되면서
    # AI 화면과 이벤트가 두 번 만들어지는 Gradio 6.5.1 문제를 피하기 위함입니다.
    @gr.render(inputs=user_page_router)
    def render_user_page(route_value: str):
        route_value = (route_value or "code").strip()
        page = route_value
        chat_seed = ""
        if route_value.startswith("{"):
            try:
                route_data = json.loads(route_value)
                page = route_data.get("page", "code")
                chat_seed = route_data.get("seed", "")
            except (json.JSONDecodeError, TypeError):
                page = "code"
        print(f"[단디봄] render_user_page: {page}")
        with gr.Column(elem_classes="app-shell"):

            if page == "code":
                gr.HTML(user_logo_html())
                gr.HTML("""
                <div style="text-align:center; margin-bottom:20px;">
                  <h1 class="page-title">안녕하세요, 어르신</h1>
                  <p class="page-subtitle">복지사님이 알려드린 인증 코드를 입력해주세요.</p>
                </div>
                """)
                code_input = gr.Textbox(
                    label="인증 코드",
                    placeholder="코드를 입력해주세요",
                    elem_classes="big-input",
                    key="code-input",
                )
                gr.Markdown("※ 데모에서는 4자리 이상 코드를 입력하면 시작됩니다.")
                code_error = gr.Markdown("", key="code-error")
                start_btn = gr.Button("시작하기", elem_classes="primary-btn", key="code-start")
                start_btn.click(
                    fn=user_verify_and_enter,
                    inputs=code_input,
                    outputs=[user_page_router, code_error, user_app_user_id_state],
                    key="code-verify-event",
                    queue=False,
                )

            elif page == "home":
                gr.HTML(user_logo_html(compact=True))
                gr.HTML("""
                <div style="text-align:center; margin-bottom:18px;">
                  <h1 class="page-title">김순자 어르신</h1>
                  <p class="page-subtitle">오늘도 단디봄이 곁에서 살펴드릴게요.</p>
                </div>
                <div class="soft-card" style="margin-bottom:14px;">
                  <strong>오늘의 상태</strong><br>
                  <span class="status-ok">● 편안한 상태예요</span>
                </div>
                """)
                user_menu_button("💊  약 복용", "medication", "home-med")
                user_menu_button("✓  컨디션 체크", "condition", "home-condition")
                user_menu_button("💬  AI 건강도우미", "chat", "home-chat")
                emergency_btn = gr.Button("☎  긴급 연락", elem_classes="danger-btn", key="home-emergency")
                emergency_btn.click(
                    fn=lambda: "emergency",
                    outputs=user_page_router,
                    key="emergency-nav-event",
                    queue=False,
                )
                user_menu_button("⚙  설정", "settings", "home-settings")

            elif page == "medication":
                user_render_header("약 복용", "home", page)
                with gr.Column(elem_classes="survey-card medication-screen"):
                    gr.HTML('<h2 class="survey-title">약은 챙겨 드셨어요?</h2>')
                    morning_med = gr.Radio(
                        choices=["✓먹었어요", "아직요"], value="✓먹었어요",
                        label="🌗 아침약", interactive=True, elem_classes="compact-radio", key="morning-med",
                    )
                    lunch_med = gr.Radio(
                        choices=["✓먹었어요", "아직요"], value="아직요",
                        label="🌕 점심약  🔒", interactive=False,
                        elem_classes="compact-radio", key="lunch-med",
                    )
                    dinner_med = gr.Radio(
                        choices=["✓먹었어요", "아직요"], value="아직요",
                        label="🌙 저녁약  🔒", interactive=False,
                        elem_classes="compact-radio", key="dinner-med",
                    )
                    gr.HTML('<p class="survey-note">※ 데모는 아침 기준으로 진행 · 점심·저녁도 동일 로직으로 동작합니다.</p>')
                    med_result = gr.Markdown("", key="med-result")
                    med_btn = gr.Button("다음 →", elem_classes=["primary-btn", "compact-next"], key="med-submit")
                    gr.HTML('<p class="survey-help">답변 후 AI가 간단히 더 여쭤볼 수 있어요</p>')
                def submit_medication(user_id, morning):
                    try:
                        assistant_message = UserBackendAPI.record_medication_and_start_chat(
                            int(user_id), morning
                        )
                    except Exception as exc:
                        print(f"[복약→AI 오류] {exc}")
                        assistant_message = "기록은 확인했어요. 잠시 뒤 다시 이야기해 주세요."
                    return user_chat_route(assistant_message)

                med_btn.click(
                    fn=submit_medication,
                    inputs=[user_app_user_id_state, morning_med],
                    outputs=user_page_router,
                    key="med-submit-event",
                    queue=True,
                )
                user_render_home_button()

            elif page == "condition":
                user_render_header("컨디션 체크", "home", page)
                with gr.Column(elem_classes="survey-card condition-screen"):
                    gr.HTML('<h2 class="survey-title">오늘 컨디션을 알려주세요</h2>')
                    meal = gr.Radio(
                        choices=["아침", "점심", "저녁"], value="아침",
                        label="🍚 식사는 하셨어요?", interactive=True, elem_classes="compact-radio", key="condition-meal",
                    )
                    gr.HTML('<p class="survey-note">※ 데모는 아침 기준으로 진행 · 점심·저녁도 동일 로직으로 동작합니다.</p>')
                    sleep = gr.Radio(
                        choices=["잠 설침", "보통", "잘잠"], value="보통",
                        label="😴 잠은 잘 주무셨어요?", interactive=True, elem_classes="compact-radio", key="condition-sleep",
                    )
                    mood = gr.Radio(
                        choices=["상쾌함", "보통", "불쾌함"], value="상쾌함",
                        label="🙂 기분은 어떠세요?", interactive=True, elem_classes="compact-radio", key="condition-mood",
                    )
                    pain = gr.Radio(
                        choices=["없음", "약간", "심각"], value="없음",
                        label="🩹 아픈 곳은 없으세요?", interactive=True, elem_classes="compact-radio", key="condition-pain",
                    )
                    condition_result = gr.Markdown("", key="condition-result")
                    submit = gr.Button("다음 →", elem_classes=["primary-btn", "compact-next"], key="condition-submit")
                    gr.HTML('<p class="survey-help">답변 후 AI가 간단히 더 여쭤볼 수 있어요</p>')
                def submit_condition(user_id, meal_v, sleep_v, mood_v, pain_v):
                    try:
                        assistant_message = UserBackendAPI.record_condition_and_start_chat(
                            int(user_id), meal_v, sleep_v, mood_v, pain_v
                        )
                    except Exception as exc:
                        print(f"[컨디션→AI 오류] {exc}")
                        assistant_message = "오늘 상태는 기록해 둘게요. 잠시 뒤 다시 이야기해 주세요."
                    return user_chat_route(assistant_message)

                submit.click(
                    fn=submit_condition,
                    inputs=[user_app_user_id_state, meal, sleep, mood, pain],
                    outputs=user_page_router,
                    key="condition-submit-event",
                    queue=True,
                )
                user_render_home_button()

            elif page == "chat":
                user_render_header("AI 건강도우미", "home", page)
                chatbot = gr.Chatbot(
                    value=[
                        {
                            "role": "assistant",
                            "content": chat_seed or "어르신, 오늘 하루는 어떻게 보내고 계세요?",
                        }
                    ],
                    height=340,
                    layout="bubble",
                    elem_classes="chat-compact",
                    key="chatbot",
                )
                message = gr.Textbox(
                    label="말씀 입력",
                    placeholder="여기에 말씀을 적어주세요",
                    elem_classes="big-input",
                    key="chat-message",
                )
                send = gr.Button("전송", elem_classes="primary-btn", key="chat-send")

                def send_message(user_id, text, chat_history):
                    text = (text or "").strip()
                    if not text:
                        return chat_history, ""
                    history_copy = list(chat_history or [])
                    history_copy.append({"role": "user", "content": text})
                    try:
                        answer = UserBackendAPI.chat(int(user_id), text)
                    except Exception as exc:
                        print(f"[AI 대화 오류] {exc}")
                        answer = "지금은 답변을 준비하지 못했어요. 잠시 뒤 다시 말씀해 주세요."

                    if user_is_demo_risk_utterance(text):
                        try:
                            alert_ready = user_activate_demo_alert(int(user_id))
                        except Exception as exc:
                            print(f"[위험 알림 DB 오류] {exc}")
                            alert_ready = False

                        if alert_ready:
                            feedback = "🚨 걱정되는 내용이 감지되어 담당 복지사에게 위험 신호를 알렸어요."
                            gr.Warning("담당 복지사에게 위험 신호 알림을 보냈습니다.")
                        else:
                            feedback = "🚨 걱정되는 내용이 감지됐어요. 필요하면 긴급 연락을 이용해 주세요."
                        answer = f"{answer}\n\n---\n{feedback}"

                    history_copy.append({"role": "assistant", "content": answer})
                    return history_copy, ""

                send.click(
                    fn=send_message,
                    inputs=[user_app_user_id_state, message, chatbot],
                    outputs=[chatbot, message],
                    key="chat-click-event",
                    queue=False,
                )
                message.submit(
                    fn=send_message,
                    inputs=[user_app_user_id_state, message, chatbot],
                    outputs=[chatbot, message],
                    key="chat-submit-event",
                    queue=False,
                )
                user_render_home_button()

            elif page == "emergency":
                user_render_header("긴급 연락", "home", page)
                gr.HTML("""
                <div style="text-align:center; padding:18px 4px;">
                  <div style="font-size:2.4rem;">☎</div>
                  <h2 class="page-title">긴급 연락을 보낼까요?</h2>
                  <p class="page-subtitle">보호자님과 복지사님께 바로 알려드릴게요.</p>
                </div>
                """)
                result = gr.Markdown("", key="emergency-result")
                contact = gr.Button("지금 연락하기", elem_classes="danger-btn", key="emergency-send")
                cancel = gr.Button("아니요, 괜찮아요", elem_classes="secondary-btn", key="emergency-cancel")
                contact.click(
                    fn=lambda: "✅ 보호자님과 복지사님께 긴급 알림을 보냈습니다.",
                    outputs=result,
                    key="emergency-send-event",
                    queue=False,
                )
                cancel.click(
                    fn=lambda: "home",
                    outputs=user_page_router,
                    key="emergency-cancel-event",
                    queue=False,
                )
                user_render_home_button()

            elif page == "settings":
                user_render_header("설정", "home", page)
                user_menu_button("내 정보", "my_info", "settings-me")
                user_menu_button("보호자 정보", "guardian_info", "settings-guardian")
                user_menu_button("복지사 정보", "worker_info", "settings-worker")
                gr.Checkbox(label="알림 받기", value=True, key="notification-setting")
                user_render_home_button()

            elif page == "my_info":
                user_render_header("내 정보", "settings", page)
                gr.HTML("""
                <div class="card">
                  <strong>이름</strong><br>김순자<br><br>
                  <strong>생년월일</strong><br>1950년 4월 12일<br><br>
                  <strong>연락처</strong><br>010-1234-5678
                </div>
                """)
                user_render_home_button()

            elif page == "guardian_info":
                user_render_header("보호자 정보", "settings", page)
                gr.HTML("""
                <div class="card">
                  <strong>보호자</strong><br>김민수 (아들)<br><br>
                  <strong>연락처</strong><br>010-9876-5432
                </div>
                """)
                call_result = gr.Markdown("", key="guardian-call-result")
                call = gr.Button("보호자에게 전화하기", elem_classes="secondary-btn", key="guardian-call")
                call.click(
                    fn=lambda: "☎ 보호자에게 전화를 연결합니다.",
                    outputs=call_result,
                    key="guardian-call-event",
                    queue=False,
                )
                user_render_home_button()

            elif page == "worker_info":
                user_render_header("복지사 정보", "settings", page)
                gr.HTML("""
                <div class="card">
                  <strong>담당 복지사</strong><br>박다정<br><br>
                  <strong>소속</strong><br>단디복지센터<br><br>
                  <strong>연락처</strong><br>053-123-4567
                </div>
                """)
                user_render_home_button()

            else:
                gr.Markdown("화면을 찾을 수 없습니다.")
                reset = gr.Button("홈으로", elem_classes="primary-btn")
                reset.click(
                    fn=lambda: "home",
                    outputs=user_page_router,
                    key="reset-home-event",
                    queue=False,
                )


# 4. 앱 실행

아래 두 셀을 실행하면 사용자 앱과 복지사 앱의 공유 URL이 각각 생성됩니다.


In [140]:
# 사용자 앱 실행
user_demo.queue().launch(
    share=True, debug=False, show_error=True,
    css=USER_CSS, allowed_paths=["/content/assets"],
    prevent_thread_lock=True,
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de99804cda7ee24076.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [141]:
# 복지사 앱 실행
worker_demo.queue().launch(
    share=True, debug=False, show_error=True,
    css=WORKER_CSS, allowed_paths=["/content/assets"],
    prevent_thread_lock=True,
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4677636bc319aafc1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
